<a href="https://colab.research.google.com/github/visionbyangelic/Neuromatch/blob/main/comp-neuro/project-algonaut/ram_optimized_encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ram test

In [ ]:
!pip install scikit-learn psutil h5py

In [ ]:
import numpy as np
import psutil
from sklearn.linear_model import RidgeCV

print(f"RAM before: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# fake data matching your real X_full and Y_target shapes
n_samples = 136805
n_features = 520
n_targets = len(np.concatenate([visual_parcels, auditory_parcels, language_parcels])) if 'visual_parcels' in dir() else 142

X_fake = np.random.randn(n_samples, n_features).astype(np.float32)
Y_fake = np.random.randn(n_samples, n_targets).astype(np.float32)

print(f"RAM after creating fake data: {psutil.virtual_memory().available / 1e9:.2f} GB free")

alphas = np.logspace(-2, 4, 8)
model = RidgeCV(alphas=alphas, alpha_per_target=True, gcv_mode='svd')
model.fit(X_fake[:117000], Y_fake[:117000, :5])  # just 5 targets, train portion only

print(f"RAM after fit: {psutil.virtual_memory().available / 1e9:.2f} GB free")
print("Success!")

RAM before: 12.40 GB free
RAM after creating fake data: 11.95 GB free
RAM after fit: 11.66 GB free
Success!


Setup & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/Neuromatch'))

['algonauts_2025_challenge_tutorial_data', 'Red NeuroTigers -- Algonauts project.gdoc']


In [ ]:
# ============================================
# SETUP & IMPORTS
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import gc
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV

BASE = '/content/drive/MyDrive/Neuromatch/algonauts_2025_challenge_tutorial_data'
assert os.path.exists(BASE), f"Can't find the folder at {BASE} — is the shortcut in My Drive?"
print("Found it ✅")
print(os.listdir(BASE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found it ✅
['trained_encoding_models', 'stimulus_features', 'algonauts_2025.competitors']


Load Stimulus Features (audio, visual, language)

In [ ]:
# ============================================
# LOAD STIMULUS FEATURES
# ============================================
def load_features(modality):
    path = os.path.join(BASE, f'stimulus_features/pca/friends/{modality}/features_train.npy')
    data = np.load(path, allow_pickle=True).item()
    print(f"{modality}: {len(data)} episodes, "
          f"example shape {data[list(data.keys())[0]].shape}")
    return data

audio_df = load_features('audio')
visual_df = load_features('visual')
language_df = load_features('language')

audio: 292 episodes, example shape (591, 20)
visual: 292 episodes, example shape (591, 250)
language: 292 episodes, example shape (591, 250)


Helper Functions (episode alignment)

In [ ]:
# ============================================
# HELPER FUNCTIONS: EPISODE ID EXTRACTION & ALIGNMENT
# ============================================
def extract_episode_id(y_key):
    match = re.search(r'task-(s\d+e\d+[a-z]?)', y_key)
    return match.group(1) if match else None

def get_aligned_xyz(episode_id, audio_dict, visual_dict, language_dict, y_file, y_key_map, delay=0):
    A = audio_dict[episode_id]
    V = visual_dict[episode_id]
    L = language_dict[episode_id]
    Y = y_file[y_key_map[episode_id]][:]

    n = min(A.shape[0], V.shape[0], L.shape[0], Y.shape[0])
    A, V, L, Y = A[:n], V[:n], L[:n], Y[:n]

    if delay > 0:
        A = A[:-delay]
        V = V[:-delay]
        L = L[:-delay]
        Y = Y[delay:]

    return A, V, L, Y

def build_all_matrices(audio_dict, visual_dict, language_dict, y_file, y_key_map, episode_order, delay=0):
    A_list, V_list, L_list, Y_list = [], [], [], []
    for ep in episode_order:
        A, V, L, Y = get_aligned_xyz(ep, audio_dict, visual_dict, language_dict, y_file, y_key_map, delay=delay)
        A_list.append(A); V_list.append(V); L_list.append(L); Y_list.append(Y)
    return np.vstack(A_list), np.vstack(V_list), np.vstack(L_list), np.vstack(Y_list)

def process_subject(subject_id, audio_dict, visual_dict, language_dict, base, delay=0):
    fmri_path = os.path.join(
        base,
        f'algonauts_2025.competitors/fmri/sub-{subject_id}/func/'
        f'sub-{subject_id}_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5'
    )
    f_sub = h5py.File(fmri_path, 'r')
    y_key_map_sub = {extract_episode_id(k): k for k in f_sub.keys()}

    x_episodes = set(audio_dict.keys())
    y_episodes = set(y_key_map_sub.keys())
    shared_sub = sorted(x_episodes & y_episodes)

    print(f"sub-{subject_id}: {len(shared_sub)} shared episodes "
          f"(X-only: {len(x_episodes - y_episodes)}, Y-only: {len(y_episodes - x_episodes)})")

    X_a, X_v, X_l, Y = build_all_matrices(
        audio_dict, visual_dict, language_dict, f_sub, y_key_map_sub, shared_sub, delay=delay
    )
    f_sub.close()
    return X_a, X_v, X_l, Y

Load Subject Data (ONLY subject '01' — this is the key RAM fix)

In [ ]:
# ============================================
# LOAD SUBJECT DATA (only the subject we're analyzing)
# ============================================
subject_ids = ['01']   # <-- only load what you actually use; add others later one at a time if needed
HRF_DELAY = 3

subject_data = {}
for sid in subject_ids:
    X_a, X_v, X_l, Y = process_subject(sid, audio_df, visual_df, language_df, BASE, delay=HRF_DELAY)
    subject_data[sid] = {'audio': X_a, 'visual': X_v, 'language': X_l, 'Y': Y}
    print(f"sub-{sid}: audio={X_a.shape}, visual={X_v.shape}, language={X_l.shape}, Y={Y.shape}")

sub-01: 292 shared episodes (X-only: 0, Y-only: 0)
sub-01: audio=(136805, 20), visual=(136805, 250), language=(136805, 250), Y=(136805, 1000)


Build Input Matrices (baseline / full / scrambled)

In [ ]:
# ============================================
# BUILD INPUT MATRICES: baseline / full / scrambled
# ============================================
def build_input_matrices(X_visual, X_audio, X_language, seed=0):
    """
    - baseline: visual + audio only
    - full: visual + audio + language
    - scrambled: visual + audio + time-shuffled language (capacity-matched control)
    """
    X_baseline = np.hstack([X_visual, X_audio])
    X_full = np.hstack([X_visual, X_audio, X_language])

    rng = np.random.default_rng(seed)
    shuffle_order = rng.permutation(X_language.shape[0])
    X_language_scrambled = X_language[shuffle_order]
    X_scrambled = np.hstack([X_visual, X_audio, X_language_scrambled])

    return X_baseline, X_full, X_scrambled

input_matrices = {}
for sid in subject_ids:
    X_v = subject_data[sid]['visual']
    X_a = subject_data[sid]['audio']
    X_l = subject_data[sid]['language']

    X_baseline, X_full, X_scrambled = build_input_matrices(X_v, X_a, X_l, seed=0)

    input_matrices[sid] = {
        'baseline': X_baseline,
        'full': X_full,
        'scrambled': X_scrambled,
        'Y': subject_data[sid]['Y'],
    }
    print(f"sub-{sid}: baseline={X_baseline.shape}, full={X_full.shape}, scrambled={X_scrambled.shape}")

sub-01: baseline=(136805, 270), full=(136805, 520), scrambled=(136805, 520)


Train/Test Split Function

In [ ]:
# ============================================
# TRAIN/TEST SPLIT
# ============================================
def time_based_split(n_TRs, test_fraction=1/7):
    n_test = int(n_TRs * test_fraction)
    train_idx = np.arange(0, n_TRs - n_test)
    test_idx = np.arange(n_TRs - n_test, n_TRs)
    return train_idx, test_idx

Model Fitting Functions (chunked, memory-safe)

In [ ]:
# ============================================
# MODEL FITTING (chunked over parcels to save RAM)
# ============================================
def compute_r_per_parcel(Y_true, Y_pred):
    Y_true_c = Y_true - Y_true.mean(axis=0, keepdims=True)
    Y_pred_c = Y_pred - Y_pred.mean(axis=0, keepdims=True)
    num = (Y_true_c * Y_pred_c).sum(axis=0)
    denom = np.sqrt((Y_true_c ** 2).sum(axis=0) * (Y_pred_c ** 2).sum(axis=0))
    denom[denom == 0] = np.nan
    return num / denom

def fit_predict_r(X, Y, train_idx, test_idx, alphas=None, chunk_size=50):
    if alphas is None:
        alphas = np.logspace(-2, 4, 20)
    X_train, X_test = X[train_idx].astype(np.float32), X[test_idx].astype(np.float32)
    Y_train, Y_test = Y[train_idx], Y[test_idx]

    n_targets = Y_train.shape[1]
    r_out = np.full(n_targets, np.nan, dtype=np.float32)

    for start in range(0, n_targets, chunk_size):
        end = min(start + chunk_size, n_targets)
        model = RidgeCV(alphas=alphas, alpha_per_target=True, gcv_mode='svd')
        model.fit(X_train, Y_train[:, start:end])
        Y_pred_chunk = model.predict(X_test)
        r_out[start:end] = compute_r_per_parcel(Y_test[:, start:end], Y_pred_chunk)
        del model, Y_pred_chunk

    return r_out

Define Parcel Groups

In [ ]:
# ============================================
# PARCEL GROUPS (visual / auditory / language regions)
# ============================================
visual_parcels = list(range(0, 81)) + list(range(500, 581))
auditory_parcels = list(range(81, 87)) + [88, 89, 90, 94, 95, 98, 100, 249, 581, 583, 585, 586, 589, 590, 591, 596, 598, 600, 604, 607, 767, 771]
language_parcels = [93, 99, 103, 133, 180, 183, 222, 234, 242, 243, 335, 346, 374, 375, 376, 377, 397, 399, 401, 403, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 431, 593, 933, 938, 944]

In [ ]:
print("X_baseline:", input_matrices[sid]['baseline'].shape, input_matrices[sid]['baseline'].dtype)
print("X_full:", input_matrices[sid]['full'].shape, input_matrices[sid]['full'].dtype)
print("X_scrambled:", input_matrices[sid]['scrambled'].shape, input_matrices[sid]['scrambled'].dtype)

X_baseline: (136805, 270) float32
X_full: (136805, 520) float32
X_scrambled: (136805, 520) float32


Main Analysis: baseline vs full vs scrambled

In [ ]:
import gc

# ============================================
# MAIN ANALYSIS: baseline vs full vs scrambled
# ============================================
sid = '01'
Y_sub = subject_data[sid]['Y']          # define Y_sub BEFORE using it
all_parcels = np.concatenate([visual_parcels, auditory_parcels, language_parcels])

Y_target = Y_sub[:, all_parcels].astype(np.float32)
train_idx, test_idx = time_based_split(Y_target.shape[0])

X_baseline = input_matrices[sid]['baseline']
X_full = input_matrices[sid]['full']
X_scrambled = input_matrices[sid]['scrambled']

r_baseline = fit_predict_r(X_baseline, Y_target, train_idx, test_idx)
gc.collect()

r_full = fit_predict_r(X_full, Y_target, train_idx, test_idx)
gc.collect()

r_scrambled = fit_predict_r(X_scrambled, Y_target, train_idx, test_idx)
gc.collect()

delta_r = r_full - r_baseline

# Explicitly delete large arrays after use to free up memory
del X_baseline, X_full, X_scrambled, Y_target
gc.collect() # Call garbage collector again after deleting

print(f"mean r_baseline: {np.nanmean(r_baseline):.3f}")
print(f"mean r_full:     {np.nanmean(r_full):.3f}")
print(f"mean r_scrambled:{np.nanmean(r_scrambled):.3f}")
print(f"mean delta_r:    {np.nanmean(delta_r):.3f}")

mean r_baseline: 0.261
mean r_full:     0.276
mean r_scrambled:0.258
mean delta_r:    0.015


ram

In [ ]:
import sys
del audio_df, visual_df, language_df
gc.collect()
print("Freed raw feature dicts")

Freed raw feature dicts


In [ ]:
import psutil
print(f"RAM used: {psutil.virtual_memory().percent}%")
print(f"RAM available: {psutil.virtual_memory().available / 1e9:.2f} GB")

RAM used: 29.9%
RAM available: 9.53 GB


baseline

In [ ]:
sid = '01'
Y_sub = subject_data[sid]['Y']
all_parcels = np.concatenate([visual_parcels, auditory_parcels, language_parcels])
Y_target = Y_sub[:, all_parcels].astype(np.float32)
train_idx, test_idx = time_based_split(Y_target.shape[0])

r_baseline = fit_predict_r(input_matrices[sid]['baseline'], Y_target, train_idx, test_idx)
gc.collect()
print(f"mean r_baseline: {np.nanmean(r_baseline):.3f}")

mean r_baseline: 0.261


Modality-Specific Predictions

In [ ]:
# ============================================
# MODALITY-SPECIFIC PREDICTIONS
# ============================================
X_v = subject_data[sid]['visual']
X_a = subject_data[sid]['audio']

print("--- Specific Modality-Parcel Predictions ---")

r_aud_to_aud = fit_predict_r(X_a, Y_sub[:, auditory_parcels].astype(np.float32), train_idx, test_idx)
print(f"mean r, audio -> auditory parcels: {np.nanmean(r_aud_to_aud):.3f}")

r_vis_to_vis = fit_predict_r(X_v, Y_sub[:, visual_parcels].astype(np.float32), train_idx, test_idx)
print(f"mean r, visual -> visual parcels:   {np.nanmean(r_vis_to_vis):.3f}")

r_vis_to_aud = fit_predict_r(X_v, Y_sub[:, auditory_parcels].astype(np.float32), train_idx, test_idx)
print(f"mean r, visual -> auditory parcels: {np.nanmean(r_vis_to_aud):.3f}")

r_aud_to_vis = fit_predict_r(X_a, Y_sub[:, visual_parcels].astype(np.float32), train_idx, test_idx)
print(f"mean r, audio -> visual parcels:   {np.nanmean(r_aud_to_vis):.3f}")

--- Specific Modality-Parcel Predictions ---
mean r, audio -> auditory parcels: 0.194
mean r, visual -> visual parcels:   0.261
mean r, visual -> auditory parcels: 0.166
mean r, audio -> visual parcels:   0.128
